In [1]:
import warnings

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

warnings.filterwarnings("ignore")

# ---------------------------------------------------------------------------
# Paramètres à adapter au jeu de données
# ---------------------------------------------------------------------------
DATA_PATH = "../data/votre_jeu_de_donnees.csv"   # <- chemin du jeu de données
TARGET_COL = "target"                    # <- variable cible
ID_COLS = ["ID"]          # <- identifiants (exclus des analyses et du modèle)

In [2]:
# ---------------------------------------------------------------------------
# Charte graphique Wavestone (reprise du template use_case2_wavestone)
# ---------------------------------------------------------------------------
COLORS = {
    "primary": "#3C18AE",       # violet profond Wavestone
    "secondary": "#441DC6",     # violet vif Wavestone
    "accent": "#8B7FF0",        # violet clair
    "neutral": "#DFD6C5",       # sable / beige
    "neutral_dark": "#C0B5B3",  # gris chaud
    "bg_light": "#EBEEFF",      # bleu-violet très clair
    "dark": "#1A1A2E",          # anthracite (titres / contraste)
    "grey": "#6B6B7B",          # texte secondaire
    "positive": "#2E8B57",      # vert (classe / modalité favorable)
    "negative": "#B23A48",      # rouge (classe / modalité défavorable)
}

SEQUENCE = [
    COLORS["primary"], COLORS["accent"], COLORS["neutral"],
    COLORS["secondary"], COLORS["neutral_dark"], COLORS["grey"],
]

TEMPLATE_LAYOUT = {
    "paper_bgcolor": "white",
    "plot_bgcolor": "white",
    "font": {"family": "Calibri, Arial, sans-serif", "size": 13, "color": COLORS["dark"]},
    "title": {"font": {"size": 20, "color": COLORS["dark"]}},
    "colorway": SEQUENCE,
}

px.defaults.template = "plotly_white"
px.defaults.color_discrete_sequence = SEQUENCE
px.defaults.color_continuous_scale = [COLORS["neutral"], COLORS["accent"], COLORS["primary"]]


def style_fig(fig: go.Figure, title: str, width: int = 900, height: int = 480) -> go.Figure:
    """Applique les codes couleurs Wavestone à une figure Plotly."""
    fig.update_layout(
        title={"text": title, "x": 0.02, "xanchor": "left"},
        template={"layout": TEMPLATE_LAYOUT},
        width=width,
        height=height,
        margin={"l": 60, "r": 40, "t": 70, "b": 60},
        legend={"orientation": "h", "yanchor": "bottom", "y": 1.02, "x": 1, "xanchor": "right"},
    )
    fig.update_xaxes(showgrid=True, gridcolor="#ECECEC", zeroline=False)
    fig.update_yaxes(showgrid=True, gridcolor="#ECECEC", zeroline=False)
    return fig

# Use case Wavestone

## Contexte et reformulation du besoin

blabla

## Démarche de travail proposé

#### I- Première analyse des features
* 1: Analyse statistique générale du dataset
* 2: Analyse univariée exploratoire pour appréhender chaque feature.
* 2: **Analyse bivariée pour avoir une première compréhension du lien entre les features et la variable cible.**
#### II- Construction d'un modèle de prédiction
* 3: Déterminer la métrique d'évaluation adaptée à notre cas d'usage.
* 4: **Déterminer un modèle V0 permettant une première prédiction, sans optimisation particulière pour commencer.**
* 5: features engineering et modèle V1

#### III- Approfondir analyse des features
* 6: Etablir un plot SHAP pour avoir les premier insights de l'impact de chaque feature sur la prédiction du modèle.

## Proposition d'approfondissement & élements de cadrage supplémentaire

#### Pistes d'approfondissement
* Utiliser tel modèle
* Etudier telle population au sein des individus de la classe 1. Etudier telle type de feature (ex: les features démographiques, ou les features marketing) et leur impact sur la prédiction

#### Cadrage supplémentaire:
* Demande d'infos métier supplémentaire sur une feature
* Précision sur l'utilisation du métier de notre modèle pour cadrer le type de prédictions. 





## 1. Dataset

### 1.1 Chargement et aperçu

In [3]:
data = pd.read_csv(DATA_PATH)


def coerce_numeric(df: pd.DataFrame, seuil: float = 0.9) -> pd.DataFrame:
    """Convertit en numérique les colonnes texte dont la quasi-totalité des valeurs sont
    numériques (gère les 'NA', ' ' et autres libellés importés en objet)."""
    for col in df.select_dtypes(include="object"):
        converted = pd.to_numeric(df[col], errors="coerce")
        if converted.notna().mean() >= seuil:
            df[col] = converted
    return df


data = coerce_numeric(data)

print(f"{data.shape[0]} lignes x {data.shape[1]} variables")
print(f"Lignes dupliquées : {data.duplicated().sum()}")
data.head()

FileNotFoundError: [Errno 2] No such file or directory: '../data/votre_jeu_de_donnees.csv'

In [ ]:
def overview(df: pd.DataFrame) -> pd.DataFrame:
    """Tableau de synthèse par variable : type, complétude, cardinalité, exemples."""
    rows = []
    for col in df.columns:
        rows.append({
            "variable": col,
            "dtype": str(df[col].dtype),
            "n_manquants": int(df[col].isna().sum()),
            "pct_manquants": round(100 * df[col].isna().mean(), 2),
            "n_modalites": int(df[col].nunique()),
            "exemples": ", ".join(map(str, df[col].dropna().unique()[:3])),
        })
    return pd.DataFrame(rows)


# Stats descriptives des variables numériques
display(data.describe().T.round(2))

# Synthèse par variable
overview(data)

### 1.2 Identification des features qualitatives et quantitatives

Objectif : constituer `col_quant` (variables quantitatives) et `col_qual` (variables qualitatives),
utilisées dans toute la suite de l'analyse.

In [ ]:
def split_quant_qual(df: pd.DataFrame, exclude: list, max_modalites: int = 10) -> tuple:
    """Sépare les colonnes en quantitatives / qualitatives.

    Heuristique : numérique et plus de `max_modalites` valeurs distinctes -> quantitative.
    À ajuster au contexte : une note de satisfaction de 0 à 10 est numérique mais se
    traite souvent comme qualitative (baisser `max_modalites` pour la basculer).
    """
    cols = [c for c in df.columns if c not in exclude]
    col_quant = [
        c for c in cols
        if pd.api.types.is_numeric_dtype(df[c]) and df[c].nunique(dropna=True) > max_modalites
    ]
    col_qual = [c for c in cols if c not in col_quant]
    return col_quant, col_qual


col_quant, col_qual = split_quant_qual(data, exclude=ID_COLS + [TARGET_COL])
print(f"col_quant ({len(col_quant)}) : {col_quant}")
print(f"col_qual  ({len(col_qual)}) : {col_qual}")

### 1.3 Valeurs manquantes et liens entre manquants

Deux lectures complémentaires :

- **volume** : taux de manquants par variable ;
- **structure** : les manquants sont-ils liés entre eux ? Si `A` est manquante, `B` l'est-elle aussi
  sur les mêmes lignes ?

  Exemple assurance : si `hasLifeInsurance` est vide, alors `lifeInsuranceMonthlyCharge` est
  probablement vide aussi ; en revanche si `hasCarInsurance` est renseignée,
  `carInsuranceMonthlyCharge` ne devrait pas être vide.

  La table `missing_links` répond exactement à ces deux questions avec
  `P(B manquant | A manquant)` et `P(B manquant | A renseigné)`.

In [ ]:
def plot_missing(df: pd.DataFrame) -> go.Figure:
    """Taux de valeurs manquantes par variable."""
    miss = df.isna().mean().mul(100)
    miss = miss[miss > 0].sort_values(ascending=True)
    if miss.empty:
        print("Aucune valeur manquante.")
        return go.Figure()
    fig = px.bar(x=miss.values, y=miss.index, orientation="h",
                 labels={"x": "% de valeurs manquantes", "y": ""})
    fig.update_traces(marker_color=COLORS["primary"], showlegend=False)
    return style_fig(fig, "Valeurs manquantes par variable (%)", width=850, height=420)


def plot_null_correlation(df: pd.DataFrame) -> go.Figure:
    """Corrélation entre indicateurs de manquants : proche de 1 = manquants conjoints,
    proche de -1 = manquants exclusifs, proche de 0 = manquants indépendants."""
    cols_nan = [c for c in df.columns if df[c].isna().any()]
    if not cols_nan:
        return go.Figure()
    corr = df[cols_nan].isna().corr()
    fig = go.Figure(go.Heatmap(
        z=corr.values, x=corr.columns, y=corr.columns,
        colorscale=[[0, COLORS["negative"]], [0.5, COLORS["bg_light"]], [1, COLORS["primary"]]],
        zmin=-1, zmax=1, text=corr.round(2).values, texttemplate="%{text}",
        textfont={"size": 10},
    ))
    fig.update_xaxes(tickangle=-45)
    return style_fig(fig, "Corrélation des manquants (variables avec NaN)", width=850, height=650)


plot_missing(data).show()
plot_null_correlation(data).show()

In [ ]:
def missing_links(df: pd.DataFrame, seuil: float = 0.5) -> pd.DataFrame:
    """Pour chaque variable A contenant des manquants, liste les variables B dont les
    manquants sont alignés sur les lignes de A (P(B manquant | A manquant) >= seuil).

    Lecture attendue :
    - P(B manquant | A manquant) élevée et P(B manquant | A renseigné) proche de 0 :
      structure de manquants **liés** (ex. montant d'une option non souscrite vide quand
      l'option est absente) -> à traiter lors du feature engineering par une modalité
      explicite ou un indicateur, pas par une imputation aveugle ;
    - une P(B manquant | A renseigné) élevée signale un problème de qualité de données
      (valeur qui devrait être renseignée mais ne l'est pas).
    """
    cols_nan = [c for c in df.columns if df[c].isna().any()]
    rows = []
    for a in cols_nan:
        mask_a = df[a].isna()
        for b in df.columns:
            if b == a:
                continue
            p_lien = df.loc[mask_a, b].isna().mean()
            if p_lien >= seuil:
                rows.append({
                    "A (manquante)": a,
                    "n_manquants_A": int(mask_a.sum()),
                    "B (liée)": b,
                    "P(B manquant | A manquant)": round(p_lien, 2),
                    "P(B manquant | A renseigné)": round(df.loc[~mask_a, b].isna().mean(), 2),
                })
    if not rows:
        print(f"Aucune liaison entre manquants au seuil de {seuil:.0%}.")
        return pd.DataFrame()
    return (
        pd.DataFrame(rows)
        .sort_values("P(B manquant | A manquant)", ascending=False)
        .reset_index(drop=True)
    )


missing_links(data, seuil=0.5)

### 1.4 Variable cible

Vérifications attendues sur une cible continue : présence de manquants (lignes à exclure),
forme de la distribution (asymétrie -> envisager une transformation log), unité et ordre de
grandeur, valeurs extrêmes.

In [ ]:
# Cible numérique : coercion au cas où, exclusion des lignes sans cible
data[TARGET_COL] = pd.to_numeric(data[TARGET_COL], errors="coerce")
n_target_nan = data[TARGET_COL].isna().sum()
if n_target_nan:
    print(f"{n_target_nan} lignes sans cible -> exclues")
    data = data.loc[data[TARGET_COL].notna()].reset_index(drop=True)


def plot_target_regression(df: pd.DataFrame, target: str) -> go.Figure:
    """Distribution de la cible continue : histogramme + box plot marginal."""
    fig = px.histogram(df, x=target, nbins=50, marginal="box")
    fig.update_traces(marker_color=COLORS["primary"], selector={"type": "histogram"})
    fig.update_layout(bargap=0.05, xaxis_title=target, yaxis_title="Nombre de lignes")
    return style_fig(fig, f"Distribution de la variable cible : {target}", width=850, height=450)


target_stats = data[TARGET_COL].describe().to_frame().round(2)
target_stats.loc["skew"] = data[TARGET_COL].skew().round(2)
display(target_stats)

plot_target_regression(data, TARGET_COL).show()

### Synthèse dataset (à compléter)

Questions auxquelles répondre ici :

- volume, doublons, qualité générale des données ;
- répartition `col_quant` / `col_qual` retenue (et ajustements de l'heuristique) ;
- structure des manquants : colonnes liées, stratégie à prévoir pour le feature engineering ;
- cible : déséquilibre des classes et implication sur le choix de la métrique (AUC PR, baseline = prévalence) ;
- cible : asymétrie / transformation éventuelle (log), outliers ;

*(Conserver uniquement la ligne utile selon le cas.)*

## 2. Analyse univariée

### 2.1 Variables qualitatives — barplots en nombre et en %

In [ ]:
def clean_labels(index) -> list:
    """Modalités lisibles (manquants explicites)."""
    return ["manquant" if pd.isna(m) else str(m) for m in index]


def plot_qual_grid(df: pd.DataFrame, cols: list, mode: str = "nb", ncols: int = 4) -> go.Figure:
    """Grille de barres horizontales par modalité : effectifs (mode='nb') ou fréquences (mode='pct')."""
    if not cols:
        print("Aucune variable qualitative.")
        return go.Figure()
    ncols = min(ncols, len(cols))
    nrows = -(-len(cols) // ncols)
    fig = make_subplots(rows=nrows, cols=ncols, subplot_titles=cols)
    for i, col in enumerate(cols):
        counts = df[col].value_counts(dropna=False)
        values = counts.values if mode == "nb" else (100 * counts / counts.sum()).round(1)
        fig.add_trace(
            go.Bar(x=values, y=clean_labels(counts.index), orientation="h",
                   marker_color=SEQUENCE[i % len(SEQUENCE)], showlegend=False),
            row=i // ncols + 1, col=i % ncols + 1,
        )
    fig.update_annotations(font={"size": 13})
    suffix = "effectifs" if mode == "nb" else "fréquences (%)"
    return style_fig(fig, f"Variables qualitatives — {suffix}", width=1250, height=280 * nrows)


plot_qual_grid(data, col_qual, mode="nb").show()
plot_qual_grid(data, col_qual, mode="pct").show()

In [ ]:
# Tableau de synthèse : effectifs et % par modalité
if col_qual:
    qual_summary = (
        pd.melt(data, value_vars=col_qual)
        .groupby(["variable", "value"], observed=True, dropna=False)
        .size().rename("nb").reset_index()
    )
    qual_summary["pct"] = (
        100 * qual_summary["nb"] / qual_summary.groupby("variable")["nb"].transform("sum")
    ).round(1)
    display(qual_summary.sort_values(["variable", "nb"], ascending=[True, False]))
else:
    print("Aucune variable qualitative.")

### 2.2 Variables quantitatives — boxplots et distributions

In [ ]:
def plot_boxplot_grid(df: pd.DataFrame, cols: list, ncols: int = 3) -> go.Figure:
    """Grille de box plots (une variable quantitative par sous-graphe)."""
    if not cols:
        print("Aucune variable quantitative.")
        return go.Figure()
    ncols = min(ncols, len(cols))
    nrows = -(-len(cols) // ncols)
    fig = make_subplots(rows=nrows, cols=ncols, subplot_titles=cols)
    for i, col in enumerate(cols):
        fig.add_trace(
            go.Box(x=df[col], marker_color=SEQUENCE[i % len(SEQUENCE)], showlegend=False),
            row=i // ncols + 1, col=i % ncols + 1,
        )
    fig.update_annotations(font={"size": 13})
    return style_fig(fig, "Box plots des variables quantitatives", width=1250, height=300 * nrows)


def plot_numeric(df: pd.DataFrame, col: str) -> go.Figure:
    """Distribution d'une variable quantitative : histogramme + box plot marginal."""
    fig = px.histogram(df, x=col, nbins=40, marginal="box")
    fig.update_traces(marker_color=COLORS["primary"], selector={"type": "histogram"})
    fig.update_layout(bargap=0.05, xaxis_title=col, yaxis_title="Nombre de lignes")
    return style_fig(fig, f"Distribution de {col}", width=850, height=420)


quant_summary = data[col_quant].describe().T
quant_summary["skew"] = data[col_quant].skew()
display(quant_summary.round(2))

plot_boxplot_grid(data, col_quant).show()
for col in col_quant:
    plot_numeric(data, col).show()

### Synthèse analyse univariée (à compléter)

Questions auxquelles répondre ici :

- modalités rares / variables constantes ou quasi constantes (candidates à l'écart) ;
- distributions multimodales (candidats à des bins métier pour l'analyse bivariée) ;
- asymétries fortes / valeurs extrêmes (transformations à prévoir pour les modèles linéaires).

## 3. Analyse bivariée — lien des features avec la cible (régression)

Objectif : quantifier le lien entre chaque feature et la **cible continue**.

- **features qualitatives** : boxplot de la cible par modalité + effet relatif à la moyenne
  globale (tableau de synthèse) ;
- **features quantitatives** : nuage de points vs cible, corrélations de Pearson (linéaire)
  et de Spearman (monotone, robuste aux relations non linéaires), matrice de corrélation.

### 3.1 Features qualitatives vs cible — boxplot par modalité

In [ ]:
def plot_qual_vs_target_grid(df: pd.DataFrame, cols: list, target: str,
                             ncols: int = 3) -> go.Figure:
    """Grille de box plots : distribution de la cible par modalité de chaque feature."""
    if not cols:
        print("Aucune variable qualitative.")
        return go.Figure()
    ncols = min(ncols, len(cols))
    nrows = -(-len(cols) // ncols)
    fig = make_subplots(rows=nrows, cols=ncols, subplot_titles=cols)
    for i, col in enumerate(cols):
        fig.add_trace(
            go.Box(x=df[col].astype(str), y=df[target],
                   marker_color=SEQUENCE[i % len(SEQUENCE)], showlegend=False),
            row=i // ncols + 1, col=i % ncols + 1,
        )
    fig.update_annotations(font={"size": 13})
    return style_fig(fig, "Distribution de la cible par modalité", width=1250, height=340 * nrows)


plot_qual_vs_target_grid(data, col_qual, TARGET_COL).show()

# Effet de chaque modalité : moyenne de la cible et écart relatif à la moyenne globale
if col_qual:
    qual_effect = (
        pd.melt(data, id_vars=[TARGET_COL], value_vars=col_qual)
        .groupby(["variable", "value"], observed=True, dropna=False)[TARGET_COL]
        .agg(["size", "mean", "median"]).round(2).reset_index()
    )
    qual_effect["effet_relatif_pct"] = (
        100 * (qual_effect["mean"] / data[TARGET_COL].mean() - 1)
    ).round(1)
    display(qual_effect.sort_values(["variable", "mean"], ascending=[True, False]))

### 3.2 Features quantitatives vs cible — nuages de points

In [ ]:
def plot_scatter_vs_target(df: pd.DataFrame, col: str, target: str) -> go.Figure:
    """Nuage de points feature vs cible (transparence pour gérer la densité de points)."""
    fig = px.scatter(df, x=col, y=target, opacity=0.35)
    fig.update_traces(marker_color=COLORS["primary"])
    fig.update_layout(xaxis_title=col, yaxis_title=target)
    return style_fig(fig, f"{target} vs {col}", width=800, height=480)


for col in col_quant:
    plot_scatter_vs_target(data, col, TARGET_COL).show()

In [ ]:
def plot_correlation_matrix(corr: pd.DataFrame) -> go.Figure:
    """Heatmap de la matrice de corrélation (palette Wavestone)."""
    fig = go.Figure(go.Heatmap(
        z=corr.values, x=corr.columns, y=corr.columns,
        colorscale=[[0, COLORS["negative"]], [0.5, COLORS["bg_light"]], [1, COLORS["primary"]]],
        zmin=-1, zmax=1, text=corr.round(2).values, texttemplate="%{text}",
        textfont={"size": 10},
    ))
    fig.update_xaxes(tickangle=-45)
    return style_fig(fig, "Matrice de corrélation (Pearson)", width=850, height=720)


corr_matrix = data[col_quant + [TARGET_COL]].corr(method="pearson")
display(corr_matrix.round(2))
plot_correlation_matrix(corr_matrix).show()

# Corrélations avec la cible : Pearson (linéaire) et Spearman (monotone)
corr_target = pd.DataFrame({
    "pearson": data[col_quant].corrwith(data[TARGET_COL], method="pearson"),
    "spearman": data[col_quant].corrwith(data[TARGET_COL], method="spearman"),
}).sort_values("pearson")

fig = go.Figure(go.Bar(
    x=corr_target["pearson"].values, y=corr_target.index, orientation="h",
    text=[f"{v:.2f}" for v in corr_target["pearson"].values], textposition="outside",
    marker_color=COLORS["primary"], showlegend=False,
))
fig.update_layout(xaxis_title="Corrélation de Pearson avec la cible", yaxis_title="")
style_fig(fig, "Corrélation des variables quantitatives avec la cible",
         width=900, height=450).show()

corr_target.round(3)

### 3.3 Liens globaux entre features

Vue d'ensemble des **redondances entre features** et des **signaux avec la cible**, par type de variable :

- **features quantitatives** : matrice de corrélation de Pearson (colonnes quantitatives,
  cible incluse quand elle est quantitative), **top 10 des paires de features les plus
  corrélées** (en valeur absolue, multicolinéarité à surveiller) et **top 10 des
  corrélations en valeur absolue avec la cible** ;
- **features qualitatives** : matrice du **V de Cramér** (0 = indépendant, 1 = lien parfait,
  cible incluse quand elle est qualitative), **top 10 des paires de features les plus
  liées** et **top 10 des V de Cramér avec la cible**.

In [ ]:
def top_abs_pairs(matrix: pd.DataFrame, n: int = 10, exclude: list = (),
                  nom_valeur: str = "valeur") -> pd.DataFrame:
    """Top n des paires de variables les plus liées, en valeur absolue, dans une matrice
    symétrique (`exclude` permet d'écarter la cible des paires entre features)."""
    cols = [c for c in matrix.columns if c not in exclude]
    sub = matrix.loc[cols, cols]
    pairs = (
        sub.where(np.triu(np.ones(sub.shape, dtype=bool), k=1))
        .stack().rename(nom_valeur).reset_index()
    )
    pairs.columns = ["variable_1", "variable_2", nom_valeur]
    pairs["valeur_abs"] = pairs[nom_valeur].abs()
    return pairs.sort_values("valeur_abs", ascending=False).head(n).reset_index(drop=True)


def top_abs_with_target(values: pd.Series, n: int = 10,
                        nom_valeur: str = "valeur") -> pd.DataFrame:
    """Top n, en valeur absolue, du lien de chaque feature avec la cible."""
    out = pd.DataFrame({nom_valeur: values})
    out["valeur_abs"] = out[nom_valeur].abs()
    return out.sort_values("valeur_abs", ascending=False).head(n)


# --- Features quantitatives : matrice de corrélation (Pearson, cible incluse)
corr_matrix_quant = data[col_quant + [TARGET_COL]].corr(method="pearson")
plot_correlation_matrix(corr_matrix_quant).show()

print("Top 10 des paires de features quantitatives les plus corrélées (en valeur absolue)")
display(top_abs_pairs(corr_matrix_quant, n=10, exclude=[TARGET_COL],
                      nom_valeur="correlation").round(3))

print(f"Top 10 des corrélations en valeur absolue avec '{TARGET_COL}'")
display(top_abs_with_target(corr_matrix_quant[TARGET_COL].drop(TARGET_COL), n=10,
                            nom_valeur="correlation").round(3))

In [ ]:
from scipy.stats import chi2_contingency


def cramers_v(x: pd.Series, y: pd.Series) -> float:
    """V de Cramér entre deux variables qualitatives (0 = indépendant, 1 = lien parfait)."""
    table = pd.crosstab(x.fillna("manquant"), y.fillna("manquant"))
    if min(table.shape) < 2:
        return 0.0  # variable constante : association non testable
    chi2 = chi2_contingency(table)[0]
    n = table.to_numpy().sum()
    return float(np.sqrt(chi2 / (n * (min(table.shape) - 1))))


def cramers_v_matrix(df: pd.DataFrame, cols: list) -> pd.DataFrame:
    """Matrice du V de Cramér entre variables qualitatives (diagonale = 1)."""
    mat = pd.DataFrame(np.eye(len(cols)), index=cols, columns=cols)
    for i, a in enumerate(cols):
        for b in cols[i + 1:]:
            v = cramers_v(df[a], df[b])
            mat.loc[a, b] = v
            mat.loc[b, a] = v
    return mat


def plot_cramers_v_matrix(mat: pd.DataFrame) -> go.Figure:
    """Heatmap de la matrice du V de Cramér (palette Wavestone, échelle 0-1)."""
    fig = go.Figure(go.Heatmap(
        z=mat.values, x=mat.columns, y=mat.columns,
        colorscale=[[0, COLORS["bg_light"]], [1, COLORS["primary"]]],
        zmin=0, zmax=1, text=mat.round(2).values, texttemplate="%{text}",
        textfont={"size": 10},
    ))
    fig.update_xaxes(tickangle=-45)
    return style_fig(fig, "Matrice du V de Cramér — features qualitatives",
                     width=850, height=720)


# --- Features qualitatives : matrice du V de Cramér ---------------------------
if col_qual:
    cramer_matrix = cramers_v_matrix(data, col_qual)
    plot_cramers_v_matrix(cramer_matrix).show()

    print("Top 10 des paires de features qualitatives les plus liées (V de Cramér)")
    display(top_abs_pairs(cramer_matrix, n=10, nom_valeur="V_cramer").round(3))

    print(f"Top 10 des V de Cramér avec '{TARGET_COL}'")
    cramer_target = pd.Series(
        {c: cramers_v(data[c], data[TARGET_COL]) for c in col_qual}
    )
    display(top_abs_with_target(cramer_target, n=10, nom_valeur="V_cramer").round(3))
else:
    print("Aucune variable qualitative.")

### Synthèse analyse bivariée (à compléter)

Questions auxquelles répondre ici :

- quelles features sont significativement liées à la cible (p-value < 0.05) et avec quelle
  force (V de Cramér / corrélation) ?
- quelles modalités / tranches se détachent du taux global ?
- quelles relations sont non monotones (bins métier à créer) ?
- quelles redondances entre features (multicolinéarité) pour la modélisation ?

## 4. Premier modèle naïf (V0)

Trois familles de modèles, **sans optimisation d'hyperparamètres ni feature engineering**,
pour établir une référence :

- **XGBoost** et **CatBoost** : gradient boosting, gèrent nativement les valeurs manquantes ;
- **régression linéaire** : référence interprétable — imputation et standardisation intégrées
  dans un pipeline scikit-learn, appliquées pli par pli (aucune fuite de données).

Protocole : one-hot sur les features qualitatives, validation croisée à 5 plis.
Métriques : **RMSE** (principale), MAE, R². Baselines de référence : prédire la moyenne
constante donne RMSE ≈ écart-type de la cible (cf. 1.4) et R² = 0.

Les colonnes dérivées de l'EDA restent dans `data` : les écarter via `DROP_COLS` si elles
dupliquent leur variable d'origine.

In [ ]:
from catboost import CatBoostRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor

# Colonnes écartées du modèle : identifiants + cible (+ colonnes dérivées redondantes)
DROP_COLS = ID_COLS + [TARGET_COL]


def prepare_X(df: pd.DataFrame, drop_cols: list) -> pd.DataFrame:
    """Matrice de features : one-hot sur les qualitatives, quantitatives telles quelles."""
    X = df.drop(columns=[c for c in drop_cols if c in df.columns])
    return pd.get_dummies(X, dtype=float)


MODELS_V0 = {
    "XGBoost": XGBRegressor(random_state=42),
    "CatBoost": CatBoostRegressor(random_state=42, verbose=0),
    "Rég. linéaire": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LinearRegression()),
    ]),
}


def evaluate_regressors(models: dict, X: pd.DataFrame, y: pd.Series,
                       n_splits: int = 5) -> pd.DataFrame:
    """Validation croisée : RMSE, MAE et R² par modèle et par pli."""
    cv = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    records = []
    for name, model in models.items():
        res = cross_validate(model, X, y, cv=cv, scoring={
            "rmse": "neg_root_mean_squared_error",
            "mae": "neg_mean_absolute_error",
            "r2": "r2",
        })
        records.append({
            "modele": name,
            "RMSE_plis": -res["test_rmse"],
            "MAE_plis": -res["test_mae"],
            "R2_plis": res["test_r2"],
        })
    return pd.DataFrame(records)


def summarize_cv(cv_records: pd.DataFrame) -> pd.DataFrame:
    """Moyenne et écart-type des métriques par modèle."""
    rows = []
    for _, rec in cv_records.iterrows():
        rows.append({
            "modele": rec["modele"],
            "RMSE_moyenne": rec["RMSE_plis"].mean(),
            "RMSE_ecart_type": rec["RMSE_plis"].std(),
            "MAE_moyenne": rec["MAE_plis"].mean(),
            "MAE_ecart_type": rec["MAE_plis"].std(),
            "R2_moyenne": rec["R2_plis"].mean(),
            "R2_ecart_type": rec["R2_plis"].std(),
        })
    return pd.DataFrame(rows).set_index("modele").round(3)


X = prepare_X(data, DROP_COLS)
y = data[TARGET_COL].astype(float)
print(f"{X.shape[0]} lignes x {X.shape[1]} variables après one-hot")
print(f"Baseline (prédiction constante = moyenne) : RMSE ≈ {y.std():.1f}, R² = 0")

cv_v0 = evaluate_regressors(MODELS_V0, X, y)
summary_v0 = summarize_cv(cv_v0)
summary_v0

In [ ]:
def plot_model_comparison_reg(summary: pd.DataFrame, title: str) -> go.Figure:
    """Trois panneaux : RMSE et MAE (plus bas = mieux) et R² (plus haut = mieux)."""
    fig = make_subplots(
        rows=1, cols=3,
        subplot_titles=["RMSE (bas = mieux)", "MAE (bas = mieux)", "R² (haut = mieux)"],
    )
    colors = [COLORS["accent"], COLORS["neutral_dark"], COLORS["primary"]]
    for metric, pos in [("RMSE", 1), ("MAE", 2), ("R2", 3)]:
        col_name = f"{metric}_moyenne"
        fig.add_trace(go.Bar(
            x=summary.index, y=summary[col_name],
            text=[f"{v:.2f}" for v in summary[col_name]], textposition="outside",
            marker_color=colors, showlegend=False,
        ), row=1, col=pos)
    fig.update_annotations(font={"size": 13})
    return style_fig(fig, title, width=1100, height=420)


plot_model_comparison_reg(summary_v0,
                          "V0 — comparaison des 3 modèles (moyenne sur 5 plis)").show()

### Synthèse V0 (à compléter)

Questions auxquelles répondre ici :

- quel modèle familial sert de référence ? (ordre de grandeur de la métrique principale
  face à sa baseline)
- les plis sont-ils homogènes (faible écart-type) ou y a-t-il une instabilité à creuser ?
- cohérence avec l'analyse bivariée : le modèle a-t-il appris les relations attendues ?

## 5. Feature engineering

Checklist guidée par les constats de l'EDA :

1. **Manquants liés** (cf. 1.3 `missing_links`) : expliciter par une modalité (`fillna`)
   ou un indicateur (`isna`) plutôt qu'imputer aveuglément ;
2. **Bins métier** (cf. 3) : transformer les relations non linéaires en features
   catégorielles lisibles ;
3. **Interactions** : croiser les variables dont les effets sont conjoints
   (cf. corrélations et valeurs SHAP de la V0) ;
4. **Écarts** : variables constantes / quasi constantes, variables redondantes
   (corrélation élevée), fuites de données potentielles ;
5. **Modèles linéaires** : imputation + échelle restent dans le pipeline, rien à gérer ici.

In [ ]:
data_v1 = data.copy()

# --- 1) Manquants liés (cf. missing_links) ----------------------------------
# Exemple : les manquants de X correspondent à une absence de souscription
# data_v1["X"] = data_v1["X"].fillna("Non souscrit")

# --- 2) Bins métier (cf. analyse bivariée) ----------------------------------
# data_v1["col_group"] = pd.cut(data_v1["col"], bins=[...], labels=[...])

# --- 3) Interactions --------------------------------------------------------
# data_v1["colA_x_colB"] = data_v1["colA"] * data_v1["colB"]

# --- 4) Variables à écarter -------------------------------------------------
FE_DROP_COLS: list = []  # ex. ["variable_constante", "variable_redondante"]

print(f"{data_v1.shape[0]} lignes x {data_v1.shape[1]} variables après feature engineering")
data_v1.head()

## 6. Modèle V1

Même protocole que la V0 — mêmes plis, mêmes métriques, toujours pas d'optimisation
d'hyperparamètres — la seule différence est le jeu de features enrichi.
On compare V0 et V1 sur la métrique principale : le gain doit venir du feature
engineering, pas d'un changement de protocole.

In [ ]:
X_v1 = prepare_X(data_v1, DROP_COLS + FE_DROP_COLS)
y = data_v1[TARGET_COL].astype(float)
print(f"{X.shape[1]} variables (V0) -> {X_v1.shape[1]} variables (V1)")

cv_v1 = evaluate_regressors(MODELS_V0, X_v1, y)
summary_v1 = summarize_cv(cv_v1)
summary_v1

In [ ]:
def plot_v0_vs_v1(summary_v0: pd.DataFrame, summary_v1: pd.DataFrame,
                  metric: str, title: str) -> go.Figure:
    """Barplot groupé : métrique principale des modèles en V0 et V1."""
    fig = go.Figure()
    for stage, summary, color in [("V0", summary_v0, COLORS["accent"]),
                                  ("V1", summary_v1, COLORS["primary"])]:
        fig.add_trace(go.Bar(
            name=stage, x=summary.index, y=summary[metric],
            text=[f"{v:.3f}" for v in summary[metric]], textposition="outside",
            marker_color=color,
        ))
    fig.update_layout(barmode="group", yaxis_title=metric)
    return style_fig(fig, title, width=850, height=420)


plot_v0_vs_v1(summary_v0, summary_v1, "RMSE_moyenne",
              "V0 vs V1 — RMSE, plus bas = mieux (moyenne sur 5 plis)").show()

### Synthèse V1 (à compléter)

Questions auxquelles répondre ici :

- quel gain apporte le feature engineering sur la métrique principale ?
- les gains sont-ils réguliers sur les 5 plis ?
- quel modèle retenir pour l'analyse SHAP et l'optimisation ?

## 7. Analyse des valeurs SHAP

SHAP décompose chaque prédiction en contributions par variable, exprimées dans l'unité de la
cible : une valeur positive augmente la prédiction, négative la diminue. Deux graphiques :

1. **beeswarm** : un point par ligne ; position horizontale = contribution, couleur = valeur
   de la variable (rouge = élevée, bleu = faible) ;
2. **importance globale** : moyenne des |SHAP| par variable.

Modèle à expliquer : celui retenu à l'issue de la V1 (`BEST_MODEL`). Les boosters sont
expliqués par `TreeExplainer`, les modèles linéaires par `LinearExplainer` (valeurs exactes) ;
pour un pipeline, l'explication se fait dans l'espace transformé (imputation + échelle).

In [ ]:
import shap

BEST_MODEL = "XGBoost"  # <- à adapter : modèle retenu à l'issue de la V1

model = MODELS_V0[BEST_MODEL]
model.fit(X_v1, y)

# Un pipeline impose d'expliquer dans l'espace transformé (imputer / scaler appliqués)
if isinstance(model, Pipeline):
    preprocessor = Pipeline(model.steps[:-1])
    X_explain = pd.DataFrame(preprocessor.transform(X_v1), columns=X_v1.columns,
                             index=X_v1.index)
    estimator = model.steps[-1][1]
else:
    X_explain = X_v1
    estimator = model

explainer = shap.Explainer(estimator, X_explain)
shap_values = explainer(X_explain)
print(f"Valeurs SHAP : {shap_values.shape[0]} lignes x {shap_values.shape[1]} variables")

shap.plots.beeswarm(shap_values, max_display=15)

In [ ]:
# Importance globale : moyenne des |SHAP| par variable
mean_abs_shap = (
    pd.Series(np.abs(shap_values.values).mean(axis=0), index=X_explain.columns,
              name="mean_abs_shap")
    .sort_values(ascending=False)
)
display(mean_abs_shap.head(15).round(3))

shap.plots.bar(shap_values, max_display=15)

### Lecture des valeurs SHAP (à compléter)

Questions auxquelles répondre ici :

- quelles variables dominent l'explication du modèle ?
- quels niveaux de chaque variable augmentent / diminuent la prédiction ?
- cohérence avec l'analyse bivariée (effets par modalité, corrélations) ?
- pistes d'itération : interactions, bins, variables à écarter (retour à l'étape 5).

# Prochaines étapes

- optimisation des hyperparamètres du modèle retenu (grille / recherche aléatoire) ;
- analyse des résidus du modèle V1 (structure résiduelle = feature engineering restant) ;
- itérations de feature engineering guidées par les valeurs SHAP ;
- questions client ouvertes : unité de la cible, signification des manquants, tolérance
  métier à l'erreur de prédiction.